In [1]:
import os
import sys
import argparse

import torch

project_root = os.path.abspath("/nas/tmddnjs33/project/G4/tps-dps_modi/tps-dps/src")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dynamics.mds import MDs
from utils.logging import Logger
from dps import DiffusionPathSampler

/home/tmddnjs33/program/anaconda3/envs/tps-dps/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other    

In [4]:
device = torch.device('cuda:3') if torch.cuda.is_available() else torch.device('cpu')

In [5]:

os.chdir("/nas/tmddnjs33/project/G4/tps-dps_modi/tps-dps")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"


In [6]:
parser = argparse.ArgumentParser()
# System Config
parser.add_argument("--date", type=str)
parser.add_argument("--seed", default=2, type=int)
parser.add_argument("--device", default="cuda", type=str)
parser.add_argument("--molecule", default="aldp", type=str)
parser.add_argument('--wandb', action='store_true', default=False)
# Logger Config
parser.add_argument("--save_dir", default="results", type=str)
# Policy Config
parser.add_argument("--bias", default="force", type=str)
# Sampling Config
parser.add_argument("--start_state", default="c5", type=str)
parser.add_argument("--end_state", default="c7ax", type=str)
parser.add_argument("--num_steps", default=1000, type=int)
parser.add_argument("--timestep", default=1, type=float)
parser.add_argument("--sigma", default=0.1, type=float)
parser.add_argument("--num_samples", default=16, type=int)
parser.add_argument("--temperature", default=300, type=float)
parser.add_argument("--friction", default=0.001, type=float)
# Training Config
parser.add_argument("--start_temperature", default=600, type=float)
parser.add_argument("--end_temperature", default=300, type=float)
parser.add_argument("--num_rollouts", default=1000, type=int)
parser.add_argument("--trains_per_rollout", default=1000, type=int)
parser.add_argument("--log_z_lr", default=1e-3, type=float)
parser.add_argument("--policy_lr", default=1e-4, type=float)
parser.add_argument("--batch_size", default=16, type=int)
parser.add_argument("--buffer_size", default=1000, type=int)
parser.add_argument("--max_grad_norm", default=1, type=int)
parser.add_argument("--control_variate", default="global", type=str)
args = parser.parse_args([
    "--molecule", "g4",
    "--start_state", "143d_Na",
    "--end_state", "1kf1_Na",
    "--num_steps", "5000",
    "--sigma", "0.5",
    "--num_rollouts", "100",
    "--num_samples", "4",
    "--batch_size", "2",
    "--bias", "scale",
    "--buffer_size", "100",
    "--end_temperature", "300",
    "--temperature", "300",
    "--date", "250709_g4Na_300K"
])

print(args)
args.training = True
args.save_dir = f"results/{args.date}"

Namespace(date='250709_g4Na_300K', seed=2, device='cuda', molecule='g4', wandb=False, save_dir='results', bias='scale', start_state='143d_Na', end_state='1kf1_Na', num_steps=5000, timestep=1, sigma=0.5, num_samples=4, temperature=300.0, friction=0.001, start_temperature=600, end_temperature=300.0, num_rollouts=100, trains_per_rollout=1000, log_z_lr=0.001, policy_lr=0.0001, batch_size=2, buffer_size=100, max_grad_norm=1, control_variate='global')


In [7]:
for name in ["policies", "positions"]:
    if not os.path.exists(f"{args.save_dir}/{name}"):
        os.makedirs(f"{args.save_dir}/{name}")
if args.wandb:
    wandb.init(project="tps-dps", config=args)
torch.manual_seed(args.seed)
mds = MDs(args)
logger = Logger(args, mds)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:04<00:00,  1.23s/it]


KeyboardInterrupt: 

In [10]:
g4 =mds.mds[0]

In [13]:
atoms = list(g4.pdb.topology.atoms())

In [19]:
ion_idx

[715, 716]

In [18]:
ion_idx = [atom.index for atom in g4.pdb.topology.atoms() if len(atom.residue) == 1]

In [5]:
agent = DiffusionPathSampler(args, mds)

/home/tmddnjs33/program/anaconda3/envs/tps-dps/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other    

In [7]:
temperatures = torch.linspace(
        args.start_temperature, args.end_temperature, args.num_rollouts
    )

In [12]:
agent.replay.forces.shape

torch.Size([100, 5001, 717, 3])